# Exercise 3 — sharpe_ratio

The Sharpe ratio is the gold standard for risk-adjusted performance. It divides average daily return by daily volatility, then annualises by multiplying by √252. A Sharpe above 1.0 is considered good; above 2.0 is excellent. A Sharpe of 0 means the strategy earned nothing after accounting for risk.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def compute_returns(df):
    return df["Close"].pct_change()

def compute_equity(returns, initial=1.0):
    return (1 + returns.fillna(0)).cumprod() * initial
def max_drawdown(equity):
    peak = equity.cummax()
    return float(((equity - peak) / peak).min())

def sharpe_ratio(returns, periods_per_year=252):
    """Annualised Sharpe ratio (risk-free rate = 0).

    Steps:
      1. clean = returns.dropna()
      2. if len(clean) == 0 or clean.std() == 0: return 0.0
      3. return float(clean.mean() / clean.std() * (periods_per_year ** 0.5))

    Dropping NaN before computing prevents the first-row NaN from distorting
    the mean and std.
    """
    # TODO: implement the 3 steps above
    return 0.0


### Checks

In [ ]:
checks = 0

# 1 — returns a float
try:
    df = _synthetic()
    r  = compute_returns(df)
    sr = sharpe_ratio(r)
    assert isinstance(sr, float), f"expected float, got {type(sr)}"
    checks += 1; print("✅ 1 sharpe_ratio returns a float")
except Exception as e:
    print("❌ 1:", e)

# 2 — all-zero returns → Sharpe = 0
try:
    zero = pd.Series([0.0] * 30)
    assert sharpe_ratio(zero) == 0.0, f"expected 0.0, got {sharpe_ratio(zero)}"
    checks += 1; print("✅ 2 all-zero returns → Sharpe = 0.0")
except Exception as e:
    print("❌ 2:", e)

# 3 — empty returns → Sharpe = 0
try:
    empty = pd.Series([], dtype=float)
    assert sharpe_ratio(empty) == 0.0, f"expected 0.0, got {sharpe_ratio(empty)}"
    checks += 1; print("✅ 3 empty returns → Sharpe = 0.0")
except Exception as e:
    print("❌ 3:", e)

# 4 — positive mean, nonzero std → positive Sharpe
try:
    good = pd.Series([0.001, 0.002, -0.0005, 0.003, 0.001] * 10)
    sr   = sharpe_ratio(good)
    assert sr > 0, f"expected positive Sharpe, got {sr}"
    checks += 1; print("✅ 4 positive mean returns → positive Sharpe")
except Exception as e:
    print("❌ 4:", e)

# 5 — higher returns / lower vol → higher Sharpe
try:
    r_good  = pd.Series([0.002] * 50 + [-0.001] * 10)
    r_noisy = pd.Series([0.002] * 50 + [-0.005] * 10)
    sr_good  = sharpe_ratio(r_good)
    sr_noisy = sharpe_ratio(r_noisy)
    assert sr_good > sr_noisy, f"lower vol should give higher Sharpe: {sr_good:.3f} vs {sr_noisy:.3f}"
    checks += 1; print("✅ 5 less volatile returns → higher Sharpe")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
